# SMS Spam Detection — Model Testing

## Objective

Test the trained Multinomial Naive Bayes model on custom SMS messages that were not part of the training or test datasets.

## Testing workflow

1. Load the saved TF-IDF vectorizer and trained model.
2. Enter custom SMS messages.
3. Transform the messages using the existing vectorizer.
4. Predict whether each message is ham or spam.
5. Display the predicted class and estimated spam probability.

**Important:** The model and vectorizer are loaded from saved files. We do not retrain the model in this notebook.

In [1]:

import joblib
import pandas as pd
from pathlib import Path

# Locate the saved model files
MODEL_DIR = Path("../models")

# Load the trained model and TF-IDF vectorizer
nb_model = joblib.load(MODEL_DIR / "multinomial_nb_model.joblib")
tfidf_vectorizer = joblib.load(MODEL_DIR / "tfidf_vectorizer.joblib")

print("Model loaded successfully!")
print("TF-IDF vectorizer loaded successfully!")


Model loaded successfully!
TF-IDF vectorizer loaded successfully!


In [2]:

def predict_sms(message):
    # Convert the input message into TF-IDF features
    message_tfidf = tfidf_vectorizer.transform([message])

    # Predict the class: 0 = Ham, 1 = Spam
    prediction = nb_model.predict(message_tfidf)[0]

    # Get the estimated probability of spam
    spam_probability = nb_model.predict_proba(message_tfidf)[0][1]

    # Convert the numeric prediction into a readable label
    label = "Spam" if prediction == 1 else "Ham"

    # Display the results
    print("Message:", message)
    print("Prediction:", label)
    print(f"Estimated spam probability: {spam_probability:.2%}")


print("Prediction function created successfully!")


Prediction function created successfully!


In [3]:

# Test a normal personal message
predict_sms("Hey, are we meeting at college tomorrow?")

print("-" * 50)

# Test a typical promotional spam message
predict_sms("Congratulations! You won a free prize. Call now to claim!")

print("-" * 50)

# Test a message with a delivery notification
predict_sms("Your package is out for delivery. Track your order online.")


Message: Hey, are we meeting at college tomorrow?
Prediction: Ham
Estimated spam probability: 0.23%
--------------------------------------------------
Message: Congratulations! You won a free prize. Call now to claim!
Prediction: Spam
Estimated spam probability: 89.54%
--------------------------------------------------
Message: Your package is out for delivery. Track your order online.
Prediction: Ham
Estimated spam probability: 11.63%


In [8]:
def test_multiple_sms(messages):
    messages_tfidf = tfidf_vectorizer.transform(messages)
    predictions = nb_model.predict(messages_tfidf)
    spam_probabilities = nb_model.predict_proba(messages_tfidf)[:, 1]

    results = pd.DataFrame({
        "Message": messages,
        "Prediction": [
            "Spam" if prediction == 1 else "Ham"
            for prediction in predictions
        ],
        "Spam Probability": spam_probabilities
    })

    return results

In [9]:
results = test_multiple_sms(test_messages)
results.style.format({"Spam Probability": "{:.2%}"})

,Message,Prediction,Spam Probability
0,"Hey, I will call you when I reach home.",Ham,0.32%
1,URGENT! You have won a cash prize. Claim it now!,Spam,83.78%
2,Your bank statement is ready for viewing.,Ham,12.84%
3,FREE entry into our weekly competition. Text WIN now!,Spam,71.67%
4,Can you send me the notes from today's class?,Ham,0.94%
5,Congratulations! You have been selected for a special offer.,Ham,34.88%
6,Your appointment is confirmed for tomorrow at 10 AM.,Ham,2.82%
7,You have been chosen to receive a free gift. Call now!,Ham,34.30%


## Final Summary

### What I implemented
- Loaded the saved Multinomial Naive Bayes model and TF-IDF vectorizer.
- Created a function to classify individual SMS messages.
- Created a function to classify multiple SMS messages in a DataFrame.
- Displayed predicted labels and estimated spam probabilities.
- Tested the model on personal messages, notifications, and promotional messages.

### Key observations
The model successfully identified several typical spam messages and ordinary personal messages. However, some promotional messages were classified as ham, demonstrating that the model has limitations.

The custom examples were used for functional testing. Because their true labels were not independently verified, they cannot be used to calculate reliable accuracy.

### Conclusion
This project implements a complete baseline SMS spam detection workflow, covering data exploration, preprocessing, TF-IDF feature engineering, Multinomial Naive Bayes training, evaluation, error analysis, and custom prediction.

Future improvements could include validation-based threshold tuning, comparing alternative classifiers, and evaluating performance on a separate labeled dataset.